In [3]:
# Introduction to data ingestion
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


In [18]:
import langchain
from langchain_core.documents import Document

print("LangChain Version:", langchain.__version__)

LangChain Version: 1.4.4


In [6]:
##create a simple document
doc = Document(
    page_content="Hello, my dog is cute",
    metadata={"source": "myfile.txt",
              "page_number": 1,
              "author": "John Doe",
              "date_created": "2023-01-01",
              "custom_field": "custom_value"}
)
print("DocumentStructure");
print("Document:", doc);
print("Page Content:", doc.page_content);
print("Metadata:", doc.metadata);
## metadata is crucial for  filtering search  results,tracking document source,providing context in responses, Debugging and Auditing)

DocumentStructure
Document: page_content='Hello, my dog is cute' metadata={'source': 'myfile.txt', 'page_number': 1, 'author': 'John Doe', 'date_created': '2023-01-01', 'custom_field': 'custom_value'}
Page Content: Hello, my dog is cute
Metadata: {'source': 'myfile.txt', 'page_number': 1, 'author': 'John Doe', 'date_created': '2023-01-01', 'custom_field': 'custom_value'}


##text files reading(.txt)

In [7]:
##crete a simple txt file for ingestion
import os
os.makedirs("data/text_files", exist_ok=True)


In [2]:
sample_text = {
    "data/text_files/python_intro.txt": """Python is a high-level, interpreted programming
     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,
      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation""",
    "data/text_files/MachineLearning_intro.txt": """Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data."""
}

for filepath, content in sample_text.items():
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Sample file created: {filepath}")

Sample file created: data/text_files/python_intro.txt
Sample file created: data/text_files/MachineLearning_intro.txt


TextLoader -Read Single File

In [8]:
from langchain_community.document_loaders import TextLoader
##Loading thhe single text file using TextLoader
loader = TextLoader("data/text_files/python_intro.txt",encoding="utf-8")
documents = loader.load()
print("Loaded Documents:", documents);
print(type(documents))
print("Page Content:", documents[0].page_content);
print("content length:", len(documents[0].page_content));

print("Metadata:", documents[0].metadata);
print("Document Structure:", documents[0]);


Loaded Documents: [Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming\n     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,\n      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation')]
<class 'list'>
Page Content: Python is a high-level, interpreted programming
     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,
      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation
content length: 309
Metadata: {'source': 'data/text_files/python_intro.txt'}
Document Structure: page_content='Python is a high-level, interpreted programming
     language known for its clear syntax and exceptional readability. Created by Guido v

C:\Users\prsha\AppData\Local\Temp\ipykernel_13800\3457052189.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


##Directory Loader-Multiple Text Files

In [10]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

## Loading all the text files in a directory using DirectoryLoader
directory_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    show_progress=False,
    loader_kwargs={"encoding": "utf-8"},
)
documents = directory_loader.load()

print("Loaded Documents:", documents)
for i, doc in enumerate(documents):
    print(f"Document {i + 1}:")
    print("Page Content:", doc.page_content)
    print("Content Length:", len(doc.page_content))
    print("Metadata:", doc.metadata)
    print("Document Structure:", doc)

Loaded Documents: [Document(metadata={'source': 'data\\text_files\\MachineLearning_intro.txt'}, page_content='Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data.'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming\n     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,\n      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation')]
Document 1:
Page Content: Machine learning is a branch of artificial intelligence tha

##Text spliting statergies


In [5]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

print(
    "Text splitters available:",
    RecursiveCharacterTextSplitter.__name__,
    CharacterTextSplitter.__name__,
    TokenTextSplitter.__name__,
)

Text splitters available: RecursiveCharacterTextSplitter CharacterTextSplitter TokenTextSplitter


In [11]:
#Method 1-character Text Spliter
text=documents[0].page_content
text

'Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data.'

In [13]:
# Method 1: Character-based splitting
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import os

# Load documents if they were not created in a previous cell
if "documents" not in globals():
    if not os.path.exists("data/text_files"):
        os.makedirs("data/text_files", exist_ok=True)
        sample_text = {
            "data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991, it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation""",
            "data/text_files/MachineLearning_intro.txt": """Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data."""
        }

        for filepath, content in sample_text.items():
            os.makedirs(os.path.dirname(filepath), exist_ok=True)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)

    directory_loader = DirectoryLoader(
        "data/text_files",
        glob="*.txt",
        loader_cls=TextLoader,
        show_progress=False,
        loader_kwargs={"encoding": "utf-8"},
    )
    documents = directory_loader.load()

if not documents:
    raise ValueError("No documents were loaded. Make sure the text files exist and the loader cell has run.")

text = documents[0].page_content
print("CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
splits = char_splitter.split_text(text)
print(f"Created {len(splits)} chunks")
print("First Split:", splits[0][:100])
print("Split Documents:", splits)
print("Last Split:", splits[-1][:100])
print("Number of Splits:", len(splits))

CHARACTER TEXT SPLITTER
Created 3 chunks
First Split: Machine learning is a branch of artificial intelligence that uses data and algorithms to let compute
Split Documents: ['Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea:', 'Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make', 'Trained models make predictions on new, unseen data.']
Last Split: Trained models make predictions on new, unseen data.
Number of Splits: 3


#Method 2:Recursive Character Splitting(Recomended)

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

if "documents" not in globals() or not documents:
    raise ValueError("No documents were loaded. Run the text loader cell first.")

text = documents[0].page_content

print("\nRECURSIVE CHARACTER TEXT SPLITTER")
recursive_char_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
recursive_splits = recursive_char_splitter.split_text(text)
print(f"Created {len(recursive_splits)} chunks")
print("First Split:", recursive_splits[0][:100])
print(recursive_splits[0])
print("--------------------")
print(recursive_splits[1])


RECURSIVE CHARACTER TEXT SPLITTER
Created 3 chunks
First Split: Machine learning is a branch of artificial intelligence that uses data and algorithms to let compute
Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea:
--------------------
Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make


#Crete a simple text without natural break Point

In [16]:
simple_text = "Hello, World This is the first chunk of text that we will split into smaller chunks using the RecursiveCharacterTextSplitter. The purpose of this example is to demonstrate how the splitter works and how it can be used to process larger documents by breaking them down into manageable pieces.. This is  sentence two and it is also quite long . This is the Sentence thrree and this is also long , this is the fourth sentence and it is also quite long . This is the fifth sentence and it is also quite long . This is the sixth sentence and it is also quite long . This is the seventh sentence and it is also quite long . This is the eighth sentence and it is also quite long . This is the ninth sentence and it is also quite long . This is the tenth sentence and it is also quite long . This is the eleventh sentence and it is also quite long . This is the twelfth sentence and it is also quite long . This is the thirteenth sentence and it is also quite long . This is the fourteenth sentence and it is also quite long . This is the fifteenth sentence and it is also quite long . This is the sixteenth sentence and it is also quite long . This is the seventeenth sentence and it is also quite long . This is the eighteenth sentence and it is also quite long . This is the nineteenth sentence and it is also quite long . This is the twentieth sentence and it is also quite long ."
splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=80,
    chunk_overlap=20,
    length_function=len,
)
chunks = splitter.split_text(simple_text)
print(f"\nSimple Text Split into {len(chunks)} chunks:\n")
for index, chunk in enumerate(chunks, start=1):
    print(f"Chunk {index} Length: {len(chunk)}")
    print(f"'{chunk}'\n")


Simple Text Split into 23 chunks:

Chunk 1 Length: 76
'Hello, World This is the first chunk of text that we will split into smaller'

Chunk 2 Length: 79
'split into smaller chunks using the RecursiveCharacterTextSplitter. The purpose'

Chunk 3 Length: 79
'The purpose of this example is to demonstrate how the splitter works and how it'

Chunk 4 Length: 78
'works and how it can be used to process larger documents by breaking them down'

Chunk 5 Length: 75
'breaking them down into manageable pieces.. This is  sentence two and it is'

Chunk 6 Length: 76
'two and it is also quite long . This is the Sentence thrree and this is also'

Chunk 7 Length: 79
'and this is also long , this is the fourth sentence and it is also quite long .'

Chunk 8 Length: 77
'also quite long . This is the fifth sentence and it is also quite long . This'

Chunk 9 Length: 79
'quite long . This is the sixth sentence and it is also quite long . This is the'

Chunk 10 Length: 75
'long . This is the seventh sentence an

#Method 3: Token Based Splitting

In [17]:
print("\n Token Text Splitter")
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
token_chunks = token_splitter.split_text(simple_text)
print(f"Created {len(token_chunks)} chunks")


 Token Text Splitter
Created 8 chunks
